<a href="https://colab.research.google.com/github/alokchoudharyguliya/Transformers/blob/main/Text_Summarization_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip3 install torch torchvision

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from spacy.tokenizer import Tokenizer
from sklearn.model_selection import train_test_split
import spacy
import pandas as pd
import numpy as np
import os, re
from nltk.corpus import stopwords
import random
from tqdm import tqdm
import math

Tokenizer using spacy

In [3]:
nlp=spacy.load('en_core_web_sm')
tokenizer=Tokenizer(nlp.vocab)

In [4]:
# Add data from files into dataframes for easier access
def create_dataframe(source_text_path,target_text_path):
  txt_files_source=[file for file in os.listdir(source_text_path) if file.endswith('.txt')]
  txt_files_target=[file for file in os.listdir(target_text_path) if file.endswith('.txt')]
  df=pd.DataFrame(columns=['headlines','text'])
  for source,target in zip(txt_files_source,txt_files_target):
    assert source==target
    source_file_path=os.path.join(source_text_path,source)
    target_file_path=os.path.join(target_text_path,target)

    # Read the content of the file
    with open(source_file_path,'r',encoding='latin-1') as file:
      source_text=file.read()
    with open(target_file_path,'r',encoding='latin-1') as file:
      target_text=file.read()
    df.loc[len(df.index)]=[source_text,target_text]
  return df

In [5]:
def check_accuracy(output,labels):
  _, predpos=(output.max(1))
  num_samples=len(labels)
  num_correct=(predpos==labels).sum()
  return (num_correct/num_samples)*100

def save_checkpoint(state,filename='weights.pth.tar'):
  print("Saving weights")
  torch.save(state,filename)

def load_checkpoint(checkpoint,model,optim):
  print("Loading weights ---- ")
  model.load_state_dict(checkpoint['state_dict'])
  optim.load_state_dict(checkpoint['optimizer'])

Kaggle Dataset Fetch

In [6]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = "pseudophoenix"
os.environ["KAGGLE_KEY"] = "7a2cb2f192719312c0d0e000fbb07af1"


In [7]:
!kaggle kernels pull kiranraghavendra/text-summarization-transformer-from-scratch

Source code downloaded to /content/text-summarization-transformer-from-scratch.ipynb


In [8]:
import kagglehub
# Download the specific dataset
base_path = kagglehub.dataset_download("pariza/bbc-news-summary")

100%|██████████| 8.91M/8.91M [00:00<00:00, 162MB/s]

Extracting files...


Extracting all the articles and summaries data

In [9]:
articles_root=f"{base_path}/BBC News Summary/News Articles"
summaries_root=f"{base_path}/BBC News Summary/Summaries"
df1 = create_dataframe(f"{articles_root}/business",f"{summaries_root}/business")
df2 = create_dataframe(f"{articles_root}/entertainment",f"{summaries_root}/entertainment")
df3 = create_dataframe(f"{articles_root}/politics",f"{summaries_root}/politics")
df4 = create_dataframe(f"{articles_root}/sport",f"{summaries_root}/sport")
df5 = create_dataframe(f"{articles_root}/tech",f"{summaries_root}/tech")

Combining all the df s into one dataframe

In [10]:
df=pd.concat([df1,df2,df3,df4,df5],ignore_index=True)

In [11]:
df.head()

,headlines,text
0,ID theft surge hits US consumers\n\nAlmost a q...,"The biggest slice of the 246,570 ID fraud case..."
1,Dollar gains on Greenspan speech\n\nThe dollar...,The dollar has hit its highest level against t...
2,Market unfazed by Aurora setback\n\nAs the Aur...,"Despite its string of bad luck, he pointed out..."
3,UK Coal plunges into deeper loss\n\nShares in ...,"UK Coal said it was making ""significant progre..."
4,Troubled Marsh under SEC scrutiny\n\nThe US st...,The US stock market regulator is investigating...


In [12]:
df=df.rename(columns={"headlines":"source_text","text":"summary_text"})
X,y=df['source_text'],df['summary_text']
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)
train_df=pd.DataFrame({'source_text':X_train,'summary_text':y_train})
test_df=pd.DataFrame({'source_text':X_test,'summary_text':y_test})

In [13]:
X_train.head(),X_test.head(), y_train.head(), y_test.head(), train_df.head(), test_df.head()

(1490    Hodges announces rugby retirement\n\nScarlets ...
 2001    'Blog' picked as word of the year\n\nThe term ...
 1572    European medal chances improve\n\nWhat have th...
 1840    Sun offers processing by the hour\n\nSun Micro...
 610     UK TV channel rapped for CSI ad\n\nTV channel ...
 Name: source_text, dtype: object,
 414     Metlife buys up Citigroup insurer\n\nUS bankin...
 420     Yukos accused of lying to court\n\nRussian oil...
 1644    Mourinho plots impressive course\n\nChelsea's ...
 416     Brazil approves bankruptcy reform\n\nA major r...
 1232    Abortion not a poll issue - Blair\n\nTony Blai...
 Name: source_text, dtype: object,
 1490    The 36-year-old, who has 54 caps, was Llanelli...
 2001    The term "blog" has been chosen as the top wor...
 1572    Diane Allahgreen has been our best hurdler for...
 1840    Sun likened grid computing to the development ...
 610     The promotion material was sent in brown envel...
 Name: summary_text, dtype: object,
 414     

In [14]:
contraction_mapping={}

In [15]:
def process_dataframe(df,tokenizer, text_cleaner_func):
  df['summary_text']=df['summary_text'].apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner_func(x))])
  df['source_text']=df['source_text'].apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner_func(x))])
  return df

In [16]:
# def text_cleaner(text):
#   # This is a placeholder for the actual text cleaning logic
#   # Based on common text summarization pipelines, this function would typically handle:
#   # - Removing special characters, punctuation, and numbers
#   # - Removing extra spaces
#   # - Lowercasing (though your current code does this separately)
#   # - Removing stopwords (if desired)
#   # - Expanding contractions (if using the contraction_mapping)
#   # For now, it just returns the text as is.
#   new_text = text.lower()
#   new_text = re.sub(r"[^a-zA-Z0-9 ]", "", new_text) # Remove special characters
#   new_text = re.sub(r"\s+", " ", new_text) # Remove extra spaces
#   return new_text

In [17]:
# def process_dataframe_text(df, tokenizer, text_cleaner_func):
#   df['source_text'] = df['source_text'].apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner_func(x))])
#   df['summary_text'] = df['summary_text'].apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner_func(x))])
#   return df

In [18]:
# # Tokenize and lowercase text using spacy
# train_df = process_dataframe_text(train_df, tokenizer, text_cleaner)
# test_df = process_dataframe_text(test_df, tokenizer, text_cleaner)

In [19]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words=set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [20]:
def text_cleaner(text):
  newString=text.lower()
  newString=newString.replace('"',"'")
  newString=re.sub(r'\([^)]*\)','',newString)
  newString=re.sub('"','',newString)
  # newString=' '.join([contraction_mapping[t] if t in contraction_mapping else t for t in newString.split(" ")])
  tokens=[w for w in newString.split() if not w in stop_words]
  return " ".join(tokens)

In [21]:
contraction_mapping = {"ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not",

                           "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not",

                           "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",

                           "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have","I'm": "I am", "I've": "I have", "i'd": "i would",

                           "i'd've": "i would have", "i'll": "i will",  "i'll've": "i will have","i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would",

                           "it'd've": "it would have", "it'll": "it will", "it'll've": "it will have","it's": "it is", "let's": "let us", "ma'am": "madam",

                           "mayn't": "may not", "might've": "might have","mightn't": "might not","mightn't've": "might not have", "must've": "must have",

                           "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have","o'clock": "of the clock",

                           "oughtn't": "ought not", "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have",

                           "she'd": "she would", "she'd've": "she would have", "she'll": "she will", "she'll've": "she will have", "she's": "she is",

                           "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have","so's": "so as",

                           "this's": "this is","that'd": "that would", "that'd've": "that would have", "that's": "that is", "there'd": "there would",

                           "there'd've": "there would have", "there's": "there is", "here's": "here is","they'd": "they would", "they'd've": "they would have",

                           "they'll": "they will", "they'll've": "they will have", "they're": "they are", "they've": "they have", "to've": "to have",

                           "wasn't": "was not", "we'd": "we would", "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have", "we're": "we are",

                           "we've": "we have", "weren't": "were not", "what'll": "what will", "what'll've": "what will have", "what're": "what are",

                           "what's": "what is", "what've": "what have", "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is",

                           "where've": "where have", "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have",

                           "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not", "won't've": "will not have",

                           "would've": "would have", "wouldn't": "would not", "wouldn't've": "would not have", "y'all": "you all",

                           "y'all'd": "you all would","y'all'd've": "you all would have","y'all're": "you all are","y'all've": "you all have",

                           "you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have",

                           "you're": "you are", "you've": "you have"}

In [22]:
def text_cleaner(text):
    if text is None:
      return ""
    if type(text) is not str:
      print(text)
      return ""
    newString = text.lower()
    newString = newString.replace('"', "'")
    newString = re.sub(r'\([^)]*\)', '', newString)
    newString = re.sub('"','', newString)
    # newString = ' '.join([contraction_mapping[t] if t in contraction_mapping else t for t in newString.split(" ")])
    newString = re.sub(r"'s\b","",newString)
    newString = re.sub("[^a-zA-Z]", " ", newString)
    tokens = [w for w in newString.split() if not w in stop_words]
    return " ".join(tokens)

In [23]:
# Tokenize and lowercase text using spacy
train_df['source_text']=train_df['source_text'].apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner(x))])
# train_df=process_dataframe(train_df,tokenizer,text_cleaner)
train_df['summary_text']=train_df['summary_text'].apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner(x))])

# Fill NaN values with empty strings before tokenization to prevent TypeError
test_df['source_text']=test_df['source_text'].fillna('').apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner(x))])
test_df['summary_text']=test_df['summary_text'].fillna('').apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner(x))])

In [24]:
train_df['source_text']=train_df['source_text'].apply(lambda x:['_START_']+x+['_END_'])
train_df['summary_text']=train_df['summary_text'].apply(lambda x:['_START_']+x+['_END_'])

test_df['source_text']=test_df['source_text'].apply(lambda x:['_START_']+x+['_END_'])
test_df['summary_text']=test_df['summary_text'].apply(lambda x:['_START_']+x+['_END_'])

In [25]:
train_df.head()

,source_text,summary_text
1490,"[_START_, hodges, announces, rugby, retirement...","[_START_, year, old, caps, llanelli, player, s..."
2001,"[_START_, blog, picked, word, year, term, blog...","[_START_, term, blog, chosen, top, word, us, d..."
1572,"[_START_, european, medal, chances, improve, e...","[_START_, diane, allahgreen, best, hurdler, ti..."
1840,"[_START_, sun, offers, processing, hour, sun, ...","[_START_, sun, likened, grid, computing, devel..."
610,"[_START_, uk, tv, channel, rapped, csi, ad, tv...","[_START_, promotion, material, sent, brown, en..."


In [28]:
# Builind vocabularies - each word has an index, note: words sorted in ascending order
all_tokens=train_df['source_text'].tolist()+train_df['summary_text'].tolist()+test_df['source_text'].tolist()+test_df['summary_text'].tolist()
source_vocab={actual_word:idx for idx, (word_num, actual_word) in enumerate(sorted(enumerate(set(token for tokens in all_tokens for token  in tokens)),key=lambda x: x[1]))}
target_vocab={actual_word: idx for idx, (word_num, actual_word) in enumerate(sorted(enumerate(set(token for tokens in all_tokens for token in tokens)), key = lambda x: x[1]))}

In [29]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("using ",device)

using  cuda


In [30]:
temp=list(sorted(source_vocab.items()))
for word, idx in temp[-5:]:
  print(word, idx)

zuluaga 27632
zurich 27633
zutons 27634
zvonareva 27635
zvyagintsev 27636


In [31]:
# Define a custom dataset class
class CustomDataset(Dataset):
  def __init__(self, source_texts, target_summaries, source_vocab, target_vocab):
    self.source_texts=source_texts
    self.target_summaries=target_summaries
    self.source_vocab=source_vocab
    self.target_vocab=target_vocab

  def __len__(self):
    return len(self.source_texts)

  def __getitem__(self,idx):
    source_text=[self.source_vocab[word] for word in self.source_texts[idx]]
    target_summary=[self.target_vocab[word] for word in self.target_summaries[idx]]
    return torch.tensor(source_text), torch.tensor(target_summary)


In [32]:
train_dataset=CustomDataset(train_df['source_text'].tolist(),train_df['summary_text'].tolist(),source_vocab, target_vocab)
test_dataset=CustomDataset(test_df['source_text'].tolist(),test_df['summary_text'].tolist(),source_vocab,target_vocab)

In [33]:
def get_max_seqlen():
  max_length=0
  for index, row in train_df.iterrows():
    # Calculate the length of the current row
    row_length=len(row['source_text'])
    # Update the maximum length if the current row length is greater
    max_length=max(max_length,row_length)
  for index, row in test_df.iterrows():
    row_length=len(row['source_text'])
    # Update the maximum length if the current row length is greater
    max_length=max(max_length,row_length)

  print("Max length in dataset ", max_length)
  return max_length

In [34]:
def collate_fn(batch):
  sources, targets=zip(*batch)
  padded_sources=pad_sequence(sources, batch_first=True)
  padded_targets=pad_sequence(targets, batch_first=True)
  return padded_sources, padded_targets

In [91]:
class MultiHeadAttention(nn.Module):
  def __init__(self, embedding_dim, num_heads):
    super(MultiHeadAttention,self).__init__()
    assert embedding_dim%num_heads==0, "embedding_dim must be divisible by num_heads"
    self.embedding_dim=embedding_dim
    self.num_heads=num_heads
    self.dim_perhead=embedding_dim//num_heads
    self.W_q=nn.Linear(embedding_dim, embedding_dim)
    self.W_k=nn.Linear(embedding_dim, embedding_dim)
    self.W_v=nn.Linear(embedding_dim, embedding_dim)
    self.W_o=nn.Linear(embedding_dim, embedding_dim)
  def scaled_dot_product_attention(self,Q,K,V, mask=None):
    # Q,K,V Shape: [Batch Size X Num_Heads X Seq_len X Dim per head]
    K=K.transpose(-2,-1)
    attn_scores=torch.matmul(Q,K)/math.sqrt(self.dim_perhead)
    if mask is not None:
      attn_scores=attn_scores.masked_fill(mask==0, -1e9)
    attn_probs=torch.softmax(attn_scores, dim=-1)
    # attn_probs Shape:
    output=torch.matmul(attn_probs, V)
    return output
  def split_heads(self, X):
    batch_size, seq_length, d_model=X.size()
    X=X.view(batch_size, seq_length, self.num_heads, self.dim_perhead)

    X=X.transpose(1,2)
    return X

  def combine_heads(self, x):
    batch_size, _ , seq_length, dim_perhead=x.size()
    x=x.transpose(1,2).contiguous()
    x=x.view(batch_size, seq_length, self.embedding_dim)
    return x

  def forward(self, Q, K, V, mask=None):
    Q=self.split_heads(self.W_q(Q))
    K=self.split_heads(self.W_k(K))
    V=self.split_heads(self.W_v(V))

    attn_output=self.scaled_dot_product_attention(Q,K,V,mask)

    output=self.W_o(self.combine_heads(attn_output))
    return output


In [92]:
class PositionWiseFeedForward(nn.Module):
  def __init__(self, d_model, d_ff):
    super(PositionWiseFeedForward,self).__init__()
    self.fc1=nn.Linear(d_model,d_ff)
    self.fc2=nn.Linear(d_ff, d_model)
  def forward(self, x):
    return self.fc2(F.relu(self.fc1(x)))

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_seq_length):
    super(PositionalEncoding,self).__init__()
    pe=torch.zeros(max_seq_length,d_model)
    position=torch.arange(0,max_seq_length,dtype=torch.float).unsqueeze(1)
    div_term=torch.exp(torch.arange(0,d_model,2).float()*-(math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(position*div_term)
    pe[:,1::2]=torch.cos(position*div_term)
    self.register_buffer('pe',pe.unsqueeze(0))
  def forward(self,x):
    return x+self.pe[:,:x.size(1)]

In [93]:
class EncoderLayer(nn.Module):
  def __init__(self,d_model,num_heads, d_ff, dropout):
    super(EncoderLayer,self).__init__()
    self.self_attn=MultiHeadAttention(d_model,num_heads)
    self.feed_forward=PositionWiseFeedForward(d_model,d_ff)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x,mask):
    attn_output=self.self_attn(x,x,x,mask)
    x=self.norm1(x+self.dropout(attn_output))
    ff_output=self.feed_forward(x)

    x=self.norm2(x+self.dropout(ff_output))
    return x


class DecoderLayer(nn.Module):
  def __init__(self,d_model, num_heads, d_ff, dropout):
    super(DecoderLayer,self).__init__()
    self.self_attn=MultiHeadAttention(d_model,num_heads)
    self.cross_attn=MultiHeadAttention(d_model, num_heads)
    self.feed_forward=PositionWiseFeedForward(d_model,d_ff)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)
    self.norm3=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x,enc_output, src_mask, tgt_mask):
    attn_output=self.self_attn(x,x,x,tgt_mask)
    x=self.norm1(x+self.dropout(attn_output))
    attn_output=self.cross_attn(x,enc_output,enc_output,src_mask)
    x=self.norm2(x+self.dropout(attn_output))
    ff_output=self.feed_forward(x)
    x=self.norm3(x+self.dropout(ff_output))
    return x

In [105]:
class Transformer(nn.Module):
  def __init__(self,src_vocab_size,tgt_vocab_size,d_model,num_heads, num_layers, d_ff, max_seq_length,drop):
    super(Transformer,self).__init__()
    self.encoder_embeddings=nn.Embedding(src_vocab_size,d_model)
    self.decoder_embedding=nn.Embedding(tgt_vocab_size,d_model)
    self.positional_encoding=PositionalEncoding(d_model,max_seq_length)
    self.encoder_layers=nn.ModuleList([EncoderLayer(d_model,num_heads, d_ff, dropout) for _ in range(num_layers)])
    self.decoder_layers=nn.ModuleList([DecoderLayer(d_model,num_heads, d_ff, dropout) for _ in range(num_layers)])
    self.fc=nn.Linear(d_model,tgt_vocab_size)
    self.dropout=nn.Dropout(dropout)

  def generate_mask(self,src,tgt):
    src_mask=(src!=0).unsqueeze(1).unsqueeze(2)
    tgt_mask=(tgt!=0).unsqueeze(1).unsqueeze(3)
    seq_length=tgt.size(1)
    nopeak_mask=(1-torch.triu(torch.ones(1,seq_length,seq_length,device=device),diagonal=1)).bool()
    tgt_mas=tgt_mask&nopeak_mask
    return src_mask, tgt_mask

  def forward(self,src,tgt):
    src_mask,tgt_mask=self.generate_mask(src,tgt)
    src_embedded=self.dropout(self.positional_encoding(self.encoder_embeddings(src)))
    tgt_embedding=self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))
    enc_output=src_embedded
    for enc_layer in self.encoder_layers:
      enc_ouput=enc_layer(enc_output,src_mask)
    dec_output=tgt_embedding
    for dec_layer in self.decoder_layers:
      dec_output=dec_layer(dec_output,enc_output,src_mask,tgt_mask)
    output=self.fc(dec_output)
    return output

In [106]:
src_vocab_size=len(source_vocab)
tgt_vocab_size=len(target_vocab)
d_model=512
num_heads=8
num_layers=6
d_ff=2048
max_seq_length=get_max_seqlen()
dropout=0.1
num_workers=2
num_epochs=10
model=Transformer(src_vocab_size,tgt_vocab_size,d_model,num_heads, num_layers, d_ff, max_seq_length, dropout)
print(model)

Max length in dataset  1982
Transformer(
  (encoder_embeddings): Embedding(27637, 512)
  (decoder_embedding): Embedding(27637, 512)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=512, out_features=512, bias=True)
        (W_k): Linear(in_features=512, out_features=512, bias=True)
        (W_v): Linear(in_features=512, out_features=512, bias=True)
        (W_o): Linear(in_features=512, out_features=512, bias=True)
      )
      (feed_forward): PositionWiseFeedForward(
        (fc1): Linear(in_features=512, out_features=2048, bias=True)
        (fc2): Linear(in_features=2048, out_features=512, bias=True)
      )
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x Decoder

Let's count the number of trainable parameters our Transformer has

In [107]:
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params)

86616565


Now we have the model, but we now need to define our loss function and the optimizer which will minimize the value of this loss function


In [108]:
criterion=nn.CrossEntropyLoss(ignore_index=0)
optimizer=optim.Adam(model.parameters(), lr=0.0001,
betas=(0.9,0.98),eps=1e-9)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=10,eta_min=0)

In [109]:
# Create Dataloaders
train_loader=DataLoader(train_dataset, batch_size=4,shuffle=True,collate_fn=collate_fn,num_workers=num_workers)
test_loader=DataLoader(test_dataset,batch_size=4,shuffle=False,collate_fn=collate_fn,num_workers=num_workers)

In [110]:
source_dummy,target_dummy=next(iter(train_loader))

In [111]:
print(source_dummy.shape,target_dummy.shape)

torch.Size([4, 258]) torch.Size([4, 140])


In [112]:
print(source_dummy[1])

tensor([    1, 14231,  6436, 11404, 19084,  6159, 18056, 26628, 19084,  6161,
          460, 10610, 19265,  1937,   504, 19195,  9184, 19123, 14231,  6436,
        21530, 26478,  3466, 21535, 17171,  7790, 11719,   757, 23773,  6159,
        27327, 11415, 12648, 19782, 25841, 14231,  6436, 25335, 23203, 26071,
        18611,   558, 12918, 26717, 21859, 19216, 11188,  4141, 14486, 15094,
        18056, 13657, 24344, 17935, 19265,  7693,  5008,  6641, 13916, 16927,
        18056,  5522,   442,  2854, 14348, 16205,  3466, 25025,  2102, 16644,
         1937,   504, 19195,  6159, 15024,  3787,  2437, 19084,   539, 16205,
         3466, 21344, 10512,  5216,  5674,  2448, 27338, 24432,  8727, 19088,
        14484, 22158, 15785,  2514,  4889, 27051, 12918,  6159, 14203, 21344,
        16221, 17935, 19265,  5885,  1666,  8728,  8623, 14608,  8543, 16205,
         3466,   290, 17083,  8801, 25219, 12768, 15093, 14231,  6391, 14025,
         4092, 13467, 15401, 15668, 25154, 25639,   558, 18164, 

In [113]:
print(torch.min(target_dummy),torch.max(target_dummy))

tensor(0) tensor(27479)


In [114]:
model.to(device)
source_dummy=source_dummy.to(device)
target_dummy=target_dummy.to(device)
print()

In [115]:
y_pred=model(source_dummy, target_dummy)
print(y_pred.shape,target_dummy.shape)

torch.Size([4, 140, 27637]) torch.Size([4, 140])


In [116]:
y_pred=y_pred.reshape(-1,len(target_vocab))
target_dummy=target_dummy.reshape(-1)
print(y_pred.shape,target_dummy.shape)

torch.Size([560, 27637]) torch.Size([560])


In [119]:
def train_loop(model,dataloader,loss_fun,optimizer,device):
  model.train()
  model.to(device)
  min_loss=None
  for epoch in range(num_epochs):
    losses=[]
    accuracies=[]
    loop=tqdm(enumerate(dataloader),total=len(dataloader),leave=True)
    for batch,(x,y) in loop:
      x=x.to(device)
      y=y.to(device)
      y_pred=model(x,y)
      loss=loss_fun(y_pred.reshape(-1,len(target_vocab)),y.reshape(-1))
      losses.append(loss.detach().item())
      accuracy=check_accuracy(y_pred.reshape(-1,len(target_vocab)),y.reshape(-1))
      accuracies.append(accuracy.detach().item())
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      scheduler.step()
      loop.set_description(f"Epoch [{epoch}/{num_epochs}]")
      loop.set_postfx(loss=loss.detach().item(),accuracy=accuracy.detach().item())
  moving_loss=sum(losses)/len(losses)
  moving_accuracy=sum(accuracies)/len(accuracies)
  checkpoint={'state_dict':model.state_dict(),'optimizer':optimizer.state_dict()}
  if min_loss==None:
    min_loss=moving_loss
    save_checkpoint(checkpoint)
  elif moving_loss<min_loss:
    min_loss=moving_loss
    save_checkpoint(checkpoint)
  print(f"Epoch {0} Loss{1}, Training Accuracy={2}".format(epoch,moving_loss, moving_accuracy))

In [121]:
def train_loop(model,dataloader,loss_fun,optimizer,device):
    model.train()
    model.to(device)
    min_loss = None
    for epoch in range(num_epochs):
        losses = []
        accuracies = []
        loop = tqdm(enumerate(dataloader), total=len(dataloader), leave=True)
        for batch,(x,y) in loop:
            # put on cuda
            x = x.to(device)
            y = y.to(device)

            # forward pass
            y_pred = model(x,y)

            # calculate loss & accuracy
            loss = loss_fun(y_pred.reshape(-1,len(target_vocab)),y.reshape(-1))
            losses.append(loss.detach().item())

            accuracy = check_accuracy(y_pred.reshape(-1,len(target_vocab)),y.reshape(-1))
            accuracies.append(accuracy.detach().item())

            # zero out prior gradients
            optimizer.zero_grad()

            # backprop
            loss.backward()

            # update weights
            optimizer.step()
            scheduler.step()

            # Update TQDM progress bar
            loop.set_description(f"Epoch [{epoch}/{num_epochs}] ")
            loop.set_postfix(loss=loss.detach().item(), accuracy=accuracy.detach().item())

        moving_loss = sum(losses) / len(losses)
        moving_accuracy = sum(accuracies) / len(accuracies)
        checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}
        # Save check point
        if min_loss == None:
            min_loss = moving_loss
            save_checkpoint(checkpoint)
        elif moving_loss < min_loss:
            min_loss = moving_loss
            save_checkpoint(checkpoint)
        print('Epoch {0} : Loss = {1} , Training Accuracy={2}'.format(epoch, moving_loss, moving_accuracy))

In [122]:
train_loop(model,train_loader,criterion,optimizer,device)

Epoch [0/10] : 100%|██████████| 445/445 [00:54<00:00,  8.18it/s, accuracy=47.2, loss=4.36]


Saving weights
Epoch 0 : Loss = 6.800713614131627 , Training Accuracy=23.92488934977001


Epoch [1/10] : 100%|██████████| 445/445 [00:53<00:00,  8.34it/s, accuracy=53.7, loss=3.11]


Saving weights
Epoch 1 : Loss = 3.758389205611154 , Training Accuracy=46.42243364741293


Epoch [2/10] : 100%|██████████| 445/445 [00:54<00:00,  8.10it/s, accuracy=45.5, loss=2.09]


...

Saving weights
Epoch 7 : Loss = 0.8927374453357096 , Training Accuracy=64.07732491653957


Epoch [8/10] : 100%|██████████| 445/445 [00:52<00:00,  8.53it/s, accuracy=46.4, loss=0.99]


Saving weights
Epoch 8 : Loss = 0.7516126555673192 , Training Accuracy=64.81063347720028


Epoch [9/10] : 100%|██████████| 445/445 [00:52<00:00,  8.45it/s, accuracy=54.1, loss=0.581]


Saving weights
Epoch 9 : Loss = 0.6372168241257078 , Training Accuracy=64.89657115721971


In [133]:
def test_loop(model,dataloader,loss_fun,device):
  model.eval()
  model.to(device)
  losses=[]
  samples,correct=0,0
  loop=tqdm(enumerate(dataloader),total=len(dataloader),leave=True)
  with torch.no_grad():
    for batch,(x,y) in loop:
      x=x.to(device)
      y=y.to(device)
      y_pred=model(x,y)
      print(x,y_pred,y)
      loss=loss_fun(y_pred.reshape(-1,len(target_vocab)),y.reshape(-1))
      losses.append(loss.detach().item())
      _,predpos=y_pred.reshape(-1,len(target_vocab)).max(1)
      samples+=len(y.reshape(-1))
      correct+=(predpos==y.reshape(-1)).sum().item()
      loop.set_postfix(loss=loss.item())
  print("Final Test Accuracy = ", 100*(correct/samples))

In [134]:
test_loop(model,test_loader,criterion,device)

  2%|▏         | 2/112 [00:00<00:14,  7.78it/s, loss=1.01]

tensor([[    1, 15621,  3436,  ...,     0,     0,     0],
        [    1, 27552,   198,  ...,     0,     0,     0],
        [    1, 16165, 18521,  ...,  4030, 21804,     0],
        [    1,  3007,  1187,  ...,     0,     0,     0]], device='cuda:0') tensor([[[-2.9640, 21.4622, -0.9225,  ..., -2.3160,  0.2236,  0.3772],
         [-4.6590, -2.3413, -2.6235,  ..., -4.9298, -1.5565, -1.0562],
         [-8.0121, -4.0993, -3.5968,  ..., -7.8422, -4.1278, -2.4483],
         ...,
         [-3.0725, -2.1076, -1.7783,  ..., -2.4492, -1.8272, -0.5855],
         [-3.1469, -1.9724, -1.7727,  ..., -2.5929, -1.8340, -0.6003],
         [-3.2629, -1.8329, -1.7591,  ..., -2.7617, -1.8584, -0.6016]],

        [[-2.7334, 22.2736, -0.5765,  ..., -1.9299,  0.4186,  0.3950],
         [-5.4921, -1.0500, -1.5822,  ..., -4.7833, -1.8943,  0.1513],
         [-5.0544, -0.0852, -1.1057,  ..., -5.0418, -1.8213, -0.6250],
         ...,
         [-2.7977, -1.9028, -1.6926,  ..., -2.3516, -1.4934, -1.0119],
         [

  4%|▎         | 4/112 [00:00<00:10, 10.16it/s, loss=1.39]

tensor([[    1, 25639, 18901,  ...,     0,     0,     0],
        [    1, 26515, 21160,  ...,  8503, 24291,     0],
        [    1, 20925, 25639,  ...,     0,     0,     0],
        [    1, 23059, 11180,  ...,     0,     0,     0]], device='cuda:0') tensor([[[-2.1456, 22.6386, -0.4244,  ..., -1.4638,  0.5678,  0.7767],
         [-5.8600, -2.0244, -3.3632,  ..., -5.3517, -2.7237, -1.1551],
         [-4.4767, -0.9582, -2.3997,  ..., -4.1587, -1.5369, -1.4842],
         ...,
         [-1.3959, -0.5672, -1.2307,  ..., -1.0951, -0.6455, -0.7025],
         [-1.2330, -0.3832, -1.1101,  ..., -0.9779, -0.4912, -0.5687],
         [-1.1883, -0.4227, -1.1159,  ..., -1.0017, -0.3970, -0.4949]],

        [[-2.4575, 22.5686, -0.6201,  ..., -1.8031,  0.4739,  0.5110],
         [-7.9651, -1.7287, -4.1031,  ..., -7.3990, -3.9055, -2.3999],
         [-3.8599, -1.4827, -2.2035,  ..., -3.9757, -1.7746, -1.2933],
         ...,
         [-5.0981, -2.8737, -2.1590,  ..., -4.5997, -2.0820, -1.9298],
         [

  5%|▌         | 6/112 [00:00<00:11,  9.22it/s, loss=0.863]

...

tensor([[    1, 25593, 15731, 18837, 17282,  6676, 13030, 16740,  5038, 13442,
         16924, 17171, 23232, 15093, 14025, 16740, 11736, 20289, 10951,  5167,
         21535, 20773, 26818,  9984, 18481, 16740,  9266, 13964,  9974,  5167,
          7235, 15093,  8447, 22513, 16740, 10301,  6670,  9724, 20224, 26192,
          7492, 21731,    55,  5313,   244, 22496, 25143, 21726, 14949,  5657,
         24992, 26170, 16740,  7487, 13498,  8981, 27032, 16626, 15093,  9983,
          5522, 17204, 25593, 21731, 27169,  5106,  1798,  4858,  9974,   386,
          7487,  3971, 16924, 25917, 21914, 18588, 11019,  9983, 14059,   276,
         16626, 26818, 18485,  5522, 19293,   134, 26818,  9987, 16896, 25650,
         23888,  8798,  6676, 14284, 10964,  6670, 23990, 15083,  7487,  8817,
          5661, 16740,  4429, 11678, 16626, 15770,  9984, 16626, 14203, 15922,
          9987, 12341, 22030,  8653, 16924, 11392, 19134,  6677, 17079, 19727,
          9987,  8981, 16183, 16310, 18478, 21731,  

 97%|█████████▋| 109/112 [00:08<00:00, 14.28it/s, loss=1.25] 

tensor([[    1, 11818,  6407, 13793, 15448, 19072, 11818, 24043, 22030, 19275,
         13793, 23252,  5025,  5552,   715, 21535,  8617, 24043,  6425, 11822,
          1897,   525, 13442,  6429, 22558, 22405, 20947, 25603,  5025, 26340,
         10046,  2428, 17556, 13988,  1941, 11822,  7206,  8111, 26669,  2209,
          5569,  1124,  1897,  4759,  9522,  8984,  5569,  1124,  8615, 21226,
          7615, 16654, 26882, 26986,   715,  4016, 23884, 20823,  7194, 20168,
          6846,  5552,   715, 21344,  8615, 24043, 25605, 14881, 19275, 11818,
         11648, 18420,  3560,  1401,  3106,   987,   635, 16205,  7194, 21344,
          2244, 27338, 20444, 25438, 15118, 19072, 11818,  6429, 17449, 17777,
         21982, 11539, 23447,  6429, 20350, 13793, 10426,  1708, 21344, 16834,
         18606, 21344, 17986, 15728,  3071,  6429, 22558, 22384, 16515,  1450,
          4199,  5179,  6090, 26725, 14224, 22055, 19275, 10712, 21344, 16548,
         10447, 26919,  8930, 13326, 18048, 20882,  

100%|██████████| 112/112 [00:08<00:00, 13.52it/s, loss=0.607]

tensor([[    1, 16080,  6163,  ...,     0,     0,     0],
        [    1,  3492, 22153,  ..., 22147, 19003,     0],
        [    1, 25639, 22030,  ...,     0,     0,     0],
        [    1,  1747,  2170,  ...,     0,     0,     0]], device='cuda:0') tensor([[[-2.6002, 22.0825, -0.6932,  ..., -1.9303,  0.2797,  0.5722],
         [-2.6506,  1.0064, -1.7045,  ..., -1.8044, -1.0872, -1.0202],
         [-0.3390,  0.0317,  1.1998,  ..., -0.1687,  0.2670,  1.0154],
         ...,
         [-2.9010, -1.3714, -1.8675,  ..., -2.5540, -1.3235, -0.4029],
         [-2.8525, -1.6103, -1.8519,  ..., -2.4844, -1.3749, -0.5123],
         [-2.7319, -1.6099, -1.7342,  ..., -2.3455, -1.3864, -0.5893]],

        [[-2.8026, 22.2369, -0.4512,  ..., -2.0741,  0.2774,  0.3487],
         [-4.6901, -2.1567, -1.1007,  ..., -4.0603, -2.4520, -0.7242],
         [ 0.6152,  1.9942,  1.1171,  ...,  0.6500,  0.4866,  1.0985],
         ...,
         [-2.5324, -1.2122, -2.0355,  ..., -2.4330, -1.1029, -1.1291],
         [

In [135]:
import torch

# Save only weights (recommended)
torch.save(model.state_dict(), "summarizer_model.pth")

In [139]:
# When using Hugging Face
# model.save_pretrained("summarizer_model")
# tokenizer.save_pretrained("summarizer_model")

In [140]:
# import shutil

# shutil.make_archive("summarizer_model", 'zip', "summarizer_model")

In [137]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "checkpoint.pth")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')